# Toy DGD: Two Uniform Blobs in 10D -> 2D Latent, 2-Component GMM

The smallest possible instance of the mechanism used everywhere else in this repo (see `dgd_training_demo.ipynb`, `dgd_test_inference.ipynb`): a decoder and per-sample latents optimized directly (no encoder), regularized by a Gaussian-mixture prior fit with [`tgmm`](https://adriansousapoza.github.io/tgmm/). Data here is synthetic and low-dimensional enough that training takes seconds and the latent space is directly plottable -- no PCA/UMAP needed for the 2D latent (only for glancing at the raw 10D data). Runs on CPU, no RAPIDS/GPU required.

Training setup mirrors `config/config.yaml` (zero-init representations, noise injection, separate decoder/representation optimizers, cosine LR schedules, GMM refit cadence) with fewer/smaller values throughout since the problem itself is much smaller -- called out in comments wherever a number differs from `config.yaml`'s. Also includes a held-out-data inference cell at the end, mirroring `dgd_test_inference.ipynb`'s Algorithm 2.

## The math

**Data** ($i = 1, \dots, N$, two uniform blobs in $\mathbb{R}^{10}$, label $y_i$ never seen by the model):

$$
x_i = c_{y_i} + u_i, \qquad u_i \sim \mathrm{Unif}([-r, r]^{10}), \qquad y_i \in \{1, 2\}
$$

**Model** -- decoder $f_\theta: \mathbb{R}^2 \to \mathbb{R}^{10}$ (a small MLP) and free per-sample latents $z_i \in \mathbb{R}^2$, all initialized at exactly $\mathbf{0}$ (`distribution: "zeros"`, same as `config.yaml`), regularized by a $K{=}2$-component Gaussian-mixture prior:

$$
\mathcal{L}(\theta, Z) = \sum_{i=1}^N \|f_\theta(\tilde z_i) - x_i\|_2^2 \;-\; \lambda \sum_{i=1}^N \log p_{\text{GMM}}(\tilde z_i), \qquad p_{\text{GMM}}(z) = \sum_{c=1}^{2} \pi_c\, \mathcal{N}(z; \mu_c, \sigma_c^2 I)
$$

where $\tilde z_i = z_i + \epsilon_i$, $\epsilon_i \sim \mathcal{N}(0, \sigma_t^2 I)$ is the same noise-injection mechanism as `noise_injection_explained.ipynb` -- essential here, not optional, since every $z_i$ starts at the *same* point $\mathbf{0}$: noise is what breaks the initial symmetry between the two blobs (see the zero-init/noise equivalence discussed in `noise_injection_explained.ipynb`).

**Optimization** -- block-coordinate, matching `DGDTrainer`: separate AdamW optimizers for $\theta$ (decoder) and $Z$ (representations), each with its own cosine-annealed learning rate ($\text{base\_lr} \to \text{final\_lr}$ over training), a reconstruction-only warm-up before the GMM term is added, and the GMM periodically refit via EM to the current (clean, un-noised) $Z$.

**A note on the sums above:** both terms use `reduction='sum'`, exactly like `trainer.py` -- the reconstruction term sums over *all* $N \times 10$ elements, the GMM term sums over $N$ per-sample log-densities. That means their relative scale depends on the data dimensionality: for FashionMNIST ($D{=}784$) reconstruction dwarfs the GMM term unless $z$'s log-density gets very large, but here ($D{=}10$) the two terms end up much closer in magnitude -- visible below in the loss curve, where the GMM term is a comparably-sized chunk of the total rather than a rounding error.

In [ ]:
import sys
import time
from pathlib import Path
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

current_dir = Path.cwd()
project_root = current_dir.parent if 'notebooks' in current_dir.parts else current_dir
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import RepresentationLayer
from src.utils.schedules import cosine_noise_schedule
from tgmm import GaussianMixture, ClusteringMetrics
from tgmm.plotting import plot_gmm

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')  # tiny problem, no need for a GPU

In [ ]:
N_per_blob = 150
dim_x = 10
r = 1.0

c1 = -3.0 * torch.ones(dim_x)
c2 = 3.0 * torch.ones(dim_x)

x1 = c1 + (torch.rand(N_per_blob, dim_x) * 2 - 1) * r
x2 = c2 + (torch.rand(N_per_blob, dim_x) * 2 - 1) * r

x = torch.cat([x1, x2], dim=0)
y_true = torch.cat([torch.zeros(N_per_blob, dtype=torch.long), torch.ones(N_per_blob, dtype=torch.long)])
N = x.shape[0]

print(f"x: {tuple(x.shape)} -- two blobs of {N_per_blob} points each in {dim_x}D, "
      f"half-width r={r}, centers at {c1[0].item():.0f}*1 and {c2[0].item():.0f}*1")

In [ ]:
pca_raw = PCA(n_components=2, random_state=42)
x_pca = pca_raw.fit_transform(x.numpy())

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_pca[:, 0], x_pca[:, 1], c=y_true.numpy(), cmap='coolwarm', s=15, alpha=0.7)
ax.set_title(f"Raw 10D data, PCA projection\n({pca_raw.explained_variance_ratio_.sum()*100:.1f}% variance explained)")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

## Model and training

Deviations from `config.yaml`, all just complexity/scale, not mechanism:

| | `config.yaml` (FashionMNIST) | here |
|---|---|---|
| `representation.n_features` | 8 | 2 (kept small on purpose -- the point of this notebook) |
| `decoder.hidden_dims` | `[128, 64]` | `[128, 64]` (same -- see note below) |
| `decoder.final_activation` | `sigmoid` (pixels in [0,1]) | identity (our data isn't bounded to [0,1]) |
| `gmm.n_components` | 20 | 2 (one per blob) |
| `gmm.covariance_type` | `tied_spherical` | `spherical` (each component gets its own variance) |
| `training.epochs` | 200 | 100 |
| `training.first_epoch_gmm` / `refit_gmm_interval` | 50 / 50 | 25 / 25 |
| everything else (`distribution: "zeros"`, optimizer betas/eps/lr, `lr_scheduler.*`, `latent_noise_*`, `lambda_gmm`, GMM `tol`/`reg_covar`/`init_*`) | -- | identical values |

Also dropped: the train/val split, checkpointing, early stopping, and best-model restoration `DGDTrainer` does -- this notebook trains on one dataset for a fixed number of epochs, and uses the inference cell at the end (on genuinely held-out data) as its generalization check instead.

**On the decoder width:** an earlier version of this notebook used a smaller `[32, 16]` decoder (fewer params felt "more toy-appropriate"), but that undershot -- reconstruction MSE plateaued noticeably above the oracle floor defined in *Reconstruction quality* below. Sweeping decoder capacity on this exact problem: `[32,16]` -> MSE 0.363, `[64,32]` -> 0.374, `[128,64]` -> 0.327 (oracle: 0.328), `[128,128,64]` -> 0.325. `config.yaml`'s own `[128, 64]` is where the gap to the oracle essentially closes, so that's what's used here too -- capacity was the actual bottleneck, not the mechanism.

In [ ]:
dim_z = 2
epochs = 100

decoder = nn.Sequential(
    nn.Linear(dim_z, 128), nn.LeakyReLU(),
    nn.Linear(128, 64), nn.LeakyReLU(),
    nn.Linear(64, dim_x),
)

rep = RepresentationLayer(
    dim=dim_z,
    n_samples=N,
    dist='zeros',
    dist_params={},
    device=device,
)

gmm = GaussianMixture(
    n_components=2,
    n_features=dim_z,
    covariance_type='spherical',
    max_iter=1000,
    tol=1e-4,
    reg_covar=1e-6,
    n_init=1,
    init_means='kmeans',
    init_weights='uniform',
    init_covariances='empirical',
    random_state=42,
    warm_start=True,
    device=device,  # explicit, matching DGDTrainer -- otherwise tgmm silently
                     # defaults to CUDA if one's available, decoupled from the
                     # rest of the model's device
)

# Decoder and representation optimizers -- same values as config.yaml's
# training.optimizer.decoder / .representation
decoder_optimizer = torch.optim.AdamW(
    decoder.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)
trainrep_optimizer = torch.optim.AdamW(
    rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)

# Cosine LR schedules, base_lr -> final_lr -- same values as config.yaml's
# training.lr_scheduler
decoder_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(decoder_optimizer, T_max=epochs, eta_min=0.001)
trainrep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trainrep_optimizer, T_max=epochs, eta_min=0.01)

print(f"Decoder: {sum(p.numel() for p in decoder.parameters())} params. Representations: {rep.n_rep} x {rep.dim}")

In [ ]:
first_epoch_gmm = 25
refit_gmm_interval = 25
lambda_gmm = 1.0
latent_noise_start = 1.0
latent_noise_end = 0.01

cluster_metrics = ClusteringMetrics()
history = {'loss': [], 'recon': [], 'gmm': [], 'noise': [], 'ami': [], 'ari': []}
epoch_times = []
frames = []  # per-epoch snapshots of z (+ GMM state once active), for the animation below
start_time = time.time()

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    # Initialize or refit the GMM -- same cadence/max_iter pattern as
    # DGDTrainer.train(): a full (re)fit at first_epoch_gmm and every
    # refit_gmm_interval epochs after, a cheap warm-started update every
    # other epoch once active.
    is_gmm_refit_epoch = epoch == first_epoch_gmm or (refit_gmm_interval and epoch % refit_gmm_interval == 0)
    current_ami, current_ari = 0.0, 0.0

    if is_gmm_refit_epoch or epoch > first_epoch_gmm:
        with torch.no_grad():
            representations = rep.z.detach()
            if is_gmm_refit_epoch:
                gmm.fit(representations, max_iter=1000 if epoch == first_epoch_gmm else 100)
            else:
                gmm.fit(representations, max_iter=100, warm_start=True)
            predicted_labels = gmm.predict(representations)
            current_ami = cluster_metrics.adjusted_mutual_info_score(y_true, predicted_labels)
            current_ari = cluster_metrics.adjusted_rand_score(y_true, predicted_labels)

    # Scheduled noise scale (cosine annealing from start to end)
    noise_scale = cosine_noise_schedule(epoch, epochs, latent_noise_start, latent_noise_end)

    decoder_optimizer.zero_grad()
    trainrep_optimizer.zero_grad()

    z = rep()  # full batch: N=300 fits trivially in memory, one step per epoch
    if noise_scale > 0:
        z = z + torch.randn_like(z) * noise_scale

    x_hat = decoder(z)
    recon_loss = F.mse_loss(x_hat, x, reduction='sum')

    if epoch >= first_epoch_gmm:
        gmm_loss = -lambda_gmm * gmm.score_samples(z).sum()
        loss = recon_loss + gmm_loss
    else:
        gmm_loss = torch.tensor(0.0)
        loss = recon_loss

    loss.backward()
    decoder_optimizer.step()
    trainrep_optimizer.step()
    decoder_scheduler.step()
    trainrep_scheduler.step()

    history['loss'].append(loss.item())
    history['recon'].append(recon_loss.item())
    history['gmm'].append(gmm_loss.item())
    history['noise'].append(noise_scale)
    history['ami'].append(current_ami)
    history['ari'].append(current_ari)

    # Snapshot this epoch's (clean, un-noised) z and the GMM's current state
    # for the training animation below -- cheap at this problem size (spherical
    # 2D covariance is just one scalar per component).
    with torch.no_grad():
        frame = {'epoch': epoch, 'z': rep().detach().clone().numpy()}
        if epoch >= first_epoch_gmm:
            frame['means'] = gmm.means_.detach().cpu().clone().numpy()
            frame['vars'] = gmm.covariances_.detach().cpu().clone().numpy()
        else:
            frame['means'], frame['vars'] = None, None
    frames.append(frame)

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    remaining_str = str(timedelta(seconds=int((epochs - epoch) * avg_epoch_time)))

    lr_decoder = decoder_optimizer.param_groups[0]['lr']
    lr_rep = trainrep_optimizer.param_groups[0]['lr']
    gmm_str = f"{gmm_loss.item():.4f}" if epoch >= first_epoch_gmm else "0.0000"
    ami_ari_str = f", AMI={current_ami:.4f}, ARI={current_ari:.4f}" if epoch >= first_epoch_gmm else ""

    print(f"Epoch {epoch}/{epochs} [Remaining: {remaining_str}, LR: Dec={lr_decoder:.2e}, Rep={lr_rep:.2e}, Noise={noise_scale:.4f}]")
    print(f"       - Loss: {loss.item():.4f}, Recon: {recon_loss.item():.4f}, GMM: {gmm_str}{ami_ari_str}")

# Final full GMM refit once training's done, for a fully-converged GMM to
# visualize (same as DGDTrainer's post-training refit -- minus the
# best-model restore step, since there's no val split here to restore from)
with torch.no_grad():
    gmm.fit(rep.z.detach(), max_iter=1000)

print(f"\nTraining completed in {str(timedelta(seconds=int(time.time() - start_time)))}")
print(f"Final GMM refit converged: {gmm.converged_} (iterations: {gmm.n_iter_})")
print(f"Final loss: {history['loss'][-1]:.4f} (recon: {history['recon'][-1]:.4f}, "
      f"AMI={history['ami'][-1]:.4f}, ARI={history['ari'][-1]:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['loss'], label='total loss')
axes[0].plot(history['recon'], label='reconstruction', alpha=0.7)
axes[0].plot(history['gmm'], label='GMM error', alpha=0.7)
axes[0].axvline(first_epoch_gmm, color='gray', linestyle='--', alpha=0.5, label='GMM term added')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Training curve')

axes[1].plot(history['noise'], color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Noise scale (sigma)')
axes[1].set_title('Noise schedule')

plt.tight_layout()
plt.show()

## Watching it train

Every epoch's $z$ (plus the GMM's state once active) was snapshotted above -- turned into a GIF below so the whole process is watchable in one go: points drifting out from the shared zero-init under noise, the GMM ellipses appearing at epoch 25 and tightening as noise decays.

In [ ]:
import io
from PIL import Image
from matplotlib.patches import Circle

z_all = np.concatenate([f['z'] for f in frames], axis=0)
pad = 0.5
xlim = (z_all[:, 0].min() - pad, z_all[:, 0].max() + pad)
ylim = (z_all[:, 1].min() - pad, z_all[:, 1].max() + pad)
cluster_colors = ['tab:purple', 'tab:red']

frame_stride = 2  # every other epoch is plenty smooth and halves the file size
selected = frames[::frame_stride]
if selected[-1]['epoch'] != frames[-1]['epoch']:
    selected.append(frames[-1])  # always include the final epoch

gif_frames = []
for f in selected:
    fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=80)
    ax.scatter(f['z'][:, 0], f['z'][:, 1], c=y_true.numpy(), cmap='coolwarm', s=10, alpha=0.7, zorder=3)
    if f['means'] is not None:
        for k in range(2):
            std = np.sqrt(f['vars'][k])
            for n_std, alpha in zip([1, 2, 3], [0.55, 0.35, 0.18]):
                ax.add_patch(Circle(f['means'][k], n_std * std, facecolor=cluster_colors[k],
                                     edgecolor='black', linewidth=1, linestyle='--', alpha=alpha, zorder=1))
            ax.scatter(*f['means'][k], color='black', marker='h', s=45, zorder=4)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel("z[0]")
    ax.set_ylabel("z[1]")
    status = "" if f['means'] is not None else "  (GMM not active yet)"
    ax.set_title(f"Epoch {f['epoch']}/{epochs}{status}", fontsize=10)
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    gif_frames.append(Image.open(buf).convert('RGB'))

# Single shared adaptive palette (built from the last, most-populated frame)
# instead of GIF's default per-frame palette -- much smaller file, no flicker.
palette_frame = gif_frames[-1].convert('P', palette=Image.ADAPTIVE, colors=64)
gif_frames_p = [f.quantize(palette=palette_frame, dither=Image.NONE) for f in gif_frames]

gif_path = Path('toy_dgd_training.gif')
gif_frames_p[0].save(
    gif_path, format='GIF', save_all=True, append_images=gif_frames_p[1:],
    duration=120, loop=0, optimize=True,
)
print(f"Saved {len(gif_frames_p)}-frame animation to {gif_path.resolve()} ({gif_path.stat().st_size / 1024:.0f} KB)")

![Training animation: z spreading out from zero-init under noise, then settling into two GMM-separated clusters](toy_dgd_training.gif)

## The learned latent space

$z$ is already 2D, so this *is* the latent space -- no PCA/UMAP projection needed (unlike the raw 10D data above, or the 8D+ latents in the main pipeline).

In [ ]:
z_final = rep().detach()

fig, ax = plt.subplots(figsize=(6, 6))
plot_gmm(
    z_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_true, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Learned 2D latent space + GMM components",
    xlabel="z[0]", ylabel="z[1]",
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
z_pred = gmm.predict(z_final)
ami = cluster_metrics.adjusted_mutual_info_score(y_true, z_pred)
ari = cluster_metrics.adjusted_rand_score(y_true, z_pred)
print(f"GMM clusters vs. true blob labels: AMI={ami:.4f}, ARI={ari:.4f} (1.0 = perfect recovery)")

## Reconstruction quality

A 2D $z$ has room to encode *which blob* a point came from, plus 2 real numbers of position within it -- the rest of each point's position across all 10 dimensions is independent uniform noise, which is information-theoretically impossible to recover from any summary, however good. So the fair comparison isn't "does $f_\theta(z_i)$ reconstruct $x_i$ exactly" (it can't), it's: how close does it get to an *oracle* that's told the true blob and just predicts that blob's center?

In [ ]:
with torch.no_grad():
    x_hat_final = decoder(z_final)

x_hat_pca = pca_raw.transform(x_hat_final.numpy())  # same fitted PCA as the raw-data plot above

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, data, title in [(axes[0], x_pca, "Original x"), (axes[1], x_hat_pca, "Reconstructed decoder(z)")]:
    ax.scatter(data[:, 0], data[:, 1], c=y_true.numpy(), cmap='coolwarm', s=15, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC 1 (of original x)")
    ax.set_ylabel("PC 2 (of original x)")
plt.tight_layout()
plt.show()

mse_model = F.mse_loss(x_hat_final, x).item()
mse_no_info = F.mse_loss(x.mean(dim=0, keepdim=True).expand_as(x), x).item()
cluster_centers = torch.stack([c1, c2])[y_true]
mse_oracle = F.mse_loss(cluster_centers, x).item()

print(f"MSE, no information (predict global mean):        {mse_no_info:.4f}")
print(f"MSE, oracle (told the true blob, predicts center): {mse_oracle:.4f}")
print(f"MSE, model reconstruction decoder(z):              {mse_model:.4f}")

## Inference on held-out data (Algorithm 2)

Same idea as `dgd_test_inference.ipynb`: freeze the trained decoder $f_\theta$ and GMM, draw fresh points from the *same* two blobs that were never used for training, and optimize only their latents:

$$
\hat z_i = \arg\min_{z} \; \|f_\theta(\tilde z) - x_i\|_2^2 \;-\; \lambda \log p_{\text{GMM}}(\tilde z), \qquad m = 1, \ldots, M
$$

starting from the same zero-init training used for $Z_0$, with a reconstruction-only warm-up ($M_0$ steps) before the GMM term is added, and the same noise schedule (mapped onto step index $m$ instead of epoch).

In [ ]:
N_test_per_blob = 50
x1_test = c1 + (torch.rand(N_test_per_blob, dim_x) * 2 - 1) * r
x2_test = c2 + (torch.rand(N_test_per_blob, dim_x) * 2 - 1) * r
x_test = torch.cat([x1_test, x2_test], dim=0)
y_test = torch.cat([torch.zeros(N_test_per_blob, dtype=torch.long), torch.ones(N_test_per_blob, dtype=torch.long)])
N_test = x_test.shape[0]

# Freeze the trained decoder -- only test_rep gets optimized below
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

test_rep = RepresentationLayer(dim=dim_z, n_samples=N_test, dist='zeros', dist_params={}, device=device)

test_optimizer = torch.optim.AdamW(
    test_rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)
M = 100                 # test optimization steps -- same as training epochs
M0 = first_epoch_gmm    # prior warm-up steps -- mirrors config.yaml's inference.prior_warmup_steps: ${training.first_epoch_gmm}
test_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(test_optimizer, T_max=M, eta_min=0.01)

print(f"Test set: {N_test} points ({N_test_per_blob} per blob), optimizing for {M} steps (warm-up: {M0})")

In [ ]:
step_history = {'loss': [], 'recon': [], 'gmm': [], 'noise': []}

for m in range(1, M + 1):
    test_optimizer.zero_grad()

    noise_scale_m = cosine_noise_schedule(m, M, latent_noise_start, latent_noise_end)

    z = test_rep()
    if noise_scale_m > 0:
        z = z + torch.randn_like(z) * noise_scale_m

    y_hat = decoder(z)
    recon_loss = F.mse_loss(y_hat, x_test, reduction='sum')

    if m >= M0:
        gmm_error = -lambda_gmm * gmm.score_samples(z).sum()
        loss = recon_loss + gmm_error
    else:
        gmm_error = torch.tensor(0.0)
        loss = recon_loss

    loss.backward()
    test_optimizer.step()
    test_scheduler.step()

    step_history['loss'].append(loss.item())
    step_history['recon'].append(recon_loss.item())
    step_history['gmm'].append(gmm_error.item())
    step_history['noise'].append(noise_scale_m)

    if m % max(1, M // 10) == 0 or m == M:
        gmm_str = f"{gmm_error.item():.4f}" if m >= M0 else "0.0000"
        print(f"Step {m}/{M} [LR: Rep={test_optimizer.param_groups[0]['lr']:.2e}, Noise={noise_scale_m:.4f}]")
        print(f"       - Loss: {loss.item():.4f}, Recon: {recon_loss.item():.4f}, GMM: {gmm_str}")

print("Test optimization complete.")

In [ ]:
z_test_final = test_rep().detach()
z_test_pred = gmm.predict(z_test_final)
test_ami = cluster_metrics.adjusted_mutual_info_score(y_test, z_test_pred)
test_ari = cluster_metrics.adjusted_rand_score(y_test, z_test_pred)
print(f"Held-out test data vs. the (frozen, trained) GMM: AMI={test_ami:.4f}, ARI={test_ari:.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
plot_gmm(
    z_test_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_test, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Held-out test latents against the trained (frozen) GMM",
    xlabel="z[0]", ylabel="z[1]",
    ax=ax,
)
plt.tight_layout()
plt.show()

## Takeaway

Same objective, same optimization recipe, same evaluation flow as the main pipeline -- decoder + representation optimizers with their own cosine LR schedules, zero-init representations broken out of symmetry by annealed noise injection, a periodically-refit GMM prior, and a separate held-out inference pass that never touches the trained decoder's weights. Scaling this up to images means: a convolutional decoder instead of an MLP, many more latent dimensions (so the latent space itself needs PCA/UMAP to look at, same as the raw 10D data here), more GMM components, and a train/val split with checkpointing and early stopping -- but the objective being optimized, and the training loop optimizing it, is exactly this one.